# Databricks Community Update — Latest Highlights

This notebook is **idempotent** — run it anytime to get a fresh one-pager visual of Databricks product updates from the last 3 months. It dynamically fetches release notes from docs.databricks.com and renders a shareable visual summary.

In [ ]:
import requests
import re
from datetime import datetime, timedelta
import calendar

# Dynamically determine the last 3 months from today
today = datetime.now()
months = []
for i in range(3):
    d = today - timedelta(days=30 * i)
    months.append((d.year, calendar.month_name[d.month]))
months.reverse()  # chronological order

print(f"Fetching Databricks release notes for: {', '.join(m[1] + ' ' + str(m[0]) for m in months)}")
print(f"Today: {today.strftime('%B %d, %Y')}")
print()

def categorize(title_lower):
    if any(k in title_lower for k in ['agent', 'genie', 'ai playground', 'mlflow', 'unity gateway', 'model serving', 'openai', 'anthropic', 'gemini', 'deepseek', 'moonshot', 'copilot']):
        return "AI & Agents"
    if any(k in title_lower for k in ['lakeflow', 'pipeline', 'zerobus', 'auto loader', 'ingest', 'cdc', 'change data', 'flow']):
        return "Data Engineering"
    if any(k in title_lower for k in ['unity catalog', 'abac', 'governance', 'sharing', 'open sharing', 'openses', 'policy', 'secret', 'classification', 'lineage']):
        return "Governance & Sharing"
    if any(k in title_lower for k in ['lakebase', 'postgres']):
        return "Lakebase / OLTP"
    if any(k in title_lower for k in ['dashboard', 'visualization', 'metric view', 'genie one', 'slack', 'excel', 'bi']):
        return "BI & Analytics"
    if any(k in title_lower for k in ['runtime', 'compute', 'serverless', 'gpu', 'environment', 'git folder', 'performance mode']):
        return "Compute & Runtime"
    if any(k in title_lower for k in ['sql function', 'arrow', 'counter', 'time_bucket', 'window measure']):
        return "SQL & Functions"
    if any(k in title_lower for k in ['apps', 'telemetry', 'session restore', 'web terminal']):
        return "Apps & Platform"
    return "Other"

def fetch_month(year, month_name):
    url = f"https://docs.databricks.com/aws/en/release-notes/product/{year}/{month_name.lower()}"
    try:
        resp = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
        if resp.status_code != 200:
            return []
        html = resp.text
        pattern = r'<h2[^>]*id=([^>]+)>(.+?)</h2>\s*<p><strong>(.+?)</strong></p>\s*<p>(.+?)</p>'
        matches = re.findall(pattern, html, re.DOTALL)
        updates = []
        for sid, title_html, date_str, desc_html in matches:
            title = re.sub(r'<!--[^>]*-->', '', title_html)
            title = re.sub(r'<[^>]+>', '', title)
            title = re.sub(r'[\u200b-\u200f\u0080-\u00ff\u2028-\u202e]', '', title)
            title = title.strip()
            desc = re.sub(r'<!--[^>]*-->', '', desc_html)
            desc = re.sub(r'<[^>]+>', '', desc).strip()
            desc = re.sub(r'\s+', ' ', desc)[:250]
            cat = categorize(title.lower())
            updates.append({
                'title': title,
                'date': date_str.strip(),
                'description': desc,
                'category': cat,
                'month': month_name,
                'url': url + '#' + sid.strip()
            })
        return updates
    except Exception as e:
        print(f"  Error fetching {month_name} {year}: {e}")
        return []

# Fetch all months
all_updates = []
for year, month_name in months:
    updates = fetch_month(year, month_name)
    print(f"  {month_name} {year}: {len(updates)} updates found")
    all_updates.extend(updates)

print(f"\nTotal updates fetched: {len(all_updates)}")

# Print category summary
from collections import Counter
cat_counts = Counter(u['category'] for u in all_updates)
print(f"\nCategory breakdown:")
for cat, count in cat_counts.most_common():
    print(f"  {cat}: {count}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter, defaultdict
from datetime import datetime

# --- ONE-PAGER VISUAL ---
fig = plt.figure(figsize=(16, 22), facecolor='white')

# Color palette
cat_colors = {
    "AI & Agents": '#FF6B6B',
    "Data Engineering": '#4ECDC4',
    "Governance & Sharing": '#45B7D1',
    "Lakebase / OLTP": '#FFA07A',
    "BI & Analytics": '#98D8C8',
    "Compute & Runtime": '#F7DC6F',
    "SQL & Functions": '#BB8FCE',
    "Apps & Platform": '#85C1E9',
    "Other": '#D5D8DC'
}

# Header
fig.text(0.5, 0.97, 'Databricks Platform Updates',
         fontsize=26, fontweight='bold', ha='center', va='top', color='#2C3E50')
period_str = f"{months[0][1]} {months[0][0]} - {months[-1][1]} {months[-1][0]}  |  {len(all_updates)} updates  |  Generated {today.strftime('%b %d, %Y')}"
fig.text(0.5, 0.948, period_str,
         fontsize=13, ha='center', va='top', color='#7F8C8D', style='italic')

# --- Top section: Category breakdown bar chart ---
ax1 = fig.add_axes([0.08, 0.76, 0.85, 0.14])
cat_counts = Counter(u['category'] for u in all_updates)
cats_sorted = cat_counts.most_common()
cat_names = [c[0] for c in cats_sorted]
cat_vals = [c[1] for c in cats_sorted]
cat_cols = [cat_colors.get(c, '#D5D8DC') for c in cat_names]
bars = ax1.barh(cat_names, cat_vals, color=cat_cols, edgecolor='white', linewidth=1.2, height=0.55)
for bar, val in zip(bars, cat_vals):
    ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=11, fontweight='bold', color='#333')
ax1.set_xlim(0, max(cat_vals) * 1.15)
ax1.set_title('Updates by Category', fontsize=14, fontweight='bold', color='#2C3E50', loc='left')
ax1.invert_yaxis()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.tick_params(axis='y', labelsize=10)
ax1.set_xlabel('Number of Updates', fontsize=9, color='#7F8C8D')

# --- Main section: Update list grouped by category ---
ax2 = fig.add_axes([0.03, 0.02, 0.94, 0.70])
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 100)
ax2.axis('off')

# Group updates by category
by_cat = defaultdict(list)
for u in all_updates:
    by_cat[u['category']].append(u)

# Sort categories by count descending
sorted_cats = sorted(by_cat.items(), key=lambda x: len(x[1]), reverse=True)

y_pos = 98
cat_num = 0
for cat, updates in sorted_cats:
    color = cat_colors.get(cat, '#D5D8DC')
    
    # Category header bar
    ax2.add_patch(mpatches.FancyBboxPatch((0.2, y_pos - 2.5), 9.6, 3.5,
                    boxstyle="round,pad=0.15", facecolor=color, edgecolor='white', alpha=0.85))
    ax2.text(0.5, y_pos - 0.8, f"{cat}  ({len(updates)})",
             fontsize=12, fontweight='bold', va='center', color='white')
    y_pos -= 4.5
    
    # List updates (max 6 per category to fit)
    for u in updates[:6]:
        # Truncate title
        title = u['title'][:95] + ('...' if len(u['title']) > 95 else '')
        ax2.text(0.6, y_pos, f"  - {title}",
                 fontsize=8.5, va='center', color='#2C3E50',
                 fontfamily='monospace')
        ax2.text(9.3, y_pos, u['date'].replace(',', '')[:12],
                 fontsize=7.5, va='center', ha='right', color='#95A5A6',
                 fontfamily='monospace')
        y_pos -= 2.8
    
    if len(updates) > 6:
        ax2.text(0.6, y_pos, f"    ... and {len(updates) - 6} more",
                 fontsize=8, va='center', color='#95A5A6', style='italic')
        y_pos -= 2.8
    
    y_pos -= 1.5
    cat_num += 1
    
    if y_pos < 5:
        break

# Footer
fig.text(0.5, 0.008, 'Source: docs.databricks.com/aws/en/release-notes/product  |  This notebook auto-refreshes data on every run',
         fontsize=8, ha='center', color='#BDC3C7', style='italic')

plt.savefig('/tmp/databricks_onepager.png', dpi=150, bbox_inches='tight', facecolor='white')
display(fig)
plt.close()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from collections import Counter

# --- TIMELINE CHART: Updates over the 3-month period ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [1, 1.5]})

# Left: Monthly update count
month_labels = [f"{m[1]} {m[0]}" for m in months]
month_counts = [sum(1 for u in all_updates if u['month'] == m[1]) for m in months]

bar_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
bars = axes[0].bar(month_labels, month_counts, color=bar_colors, edgecolor='white', linewidth=1.5, width=0.5)
for bar, count in zip(bars, month_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(count), ha='center', fontsize=14, fontweight='bold', color='#333')
axes[0].set_title('Updates per Month', fontsize=14, fontweight='bold', color='#2C3E50')
axes[0].set_ylabel('Number of Updates', fontsize=11, color='#7F8C8D')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].set_ylim(0, max(month_counts) * 1.25 if month_counts else 10)
axes[0].tick_params(axis='x', rotation=15)

# Right: Category pie chart
cat_counts = Counter(u['category'] for u in all_updates)
cats = list(cat_counts.keys())
counts = list(cat_counts.values())
colors_pie = [cat_colors.get(c, '#D5D8DC') for c in cats]

wedges, texts, autotexts = axes[1].pie(counts, labels=None, autopct='%1.0f%%',
    colors=colors_pie, startangle=90, pctdistance=0.8, textprops={'fontsize': 10, 'fontweight': 'bold'})
axes[1].set_title('Category Distribution', fontsize=14, fontweight='bold', color='#2C3E50')

# Legend
legend_labels = [f"{c} ({n})" for c, n in zip(cats, counts)]
axes[1].legend(wedges, legend_labels, loc='center left', bbox_to_anchor=(1, 0.5),
               fontsize=9, frameon=False)

plt.tight_layout()
plt.savefig('/tmp/databricks_breakdown.png', dpi=150, bbox_inches='tight', facecolor='white')
display(fig)
plt.close()

In [ ]:
# --- DETAILED UPDATE TABLE ---
# Display all updates in a readable table for reference
import pandas as pd

df_updates = pd.DataFrame(all_updates)
df_display = df_updates[['date', 'category', 'title', 'month']].copy()
df_display.columns = ['Date', 'Category', 'Update', 'Month']
df_display = df_display.sort_values(['Date'], ascending=False).reset_index(drop=True)
df_display.index += 1
df_display.index.name = '#'

print(f"Total updates: {len(df_display)}")
display(df_display)

## How to Use This Notebook

### Idempotent by Design
Every run fetches the latest release notes from `docs.databricks.com` for the trailing 3 months. No hardcoded data — the visual updates automatically as Databricks publishes new release notes.

### Sharing with Your Community

* **Export as HTML:** File -> Export -> HTML for a standalone visual page
* **Screenshots:** Right-click any chart to save as PNG for social media
* **Direct link:** Share the notebook URL with workspace members
* **Scheduled task:** A weekly Genie Code task runs every Monday at 9 AM IST to surface new updates

### Data Source
All updates are fetched from the official Databricks product release notes:
`https://docs.databricks.com/aws/en/release-notes/product/`

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter, defaultdict

# --- SHAREABLE POSTER IMAGE ---
fig = plt.figure(figsize=(16, 20), facecolor='#FAFAFA')

# Color palette
cat_colors = {
    "AI & Agents": '#FF6B6B',
    "Data Engineering": '#4ECDC4',
    "Governance & Sharing": '#45B7D1',
    "Lakebase / OLTP": '#FFA07A',
    "BI & Analytics": '#98D8C8',
    "Compute & Runtime": '#F7DC6F',
    "SQL & Functions": '#BB8FCE',
    "Apps & Platform": '#85C1E9',
    "Other": '#D5D8DC'
}

# ========== HEADER BAND ==========
header_bg = mpatches.FancyBboxPatch((0.015, 0.89), 0.97, 0.095,
    boxstyle="round,pad=0.005", facecolor='#2C3E50', edgecolor='none',
    transform=fig.transFigure, zorder=2)
fig.patches.append(header_bg)

fig.text(0.5, 0.962, 'Databricks Platform Updates',
         fontsize=28, fontweight='bold', ha='center', va='top',
         color='white', transform=fig.transFigure, zorder=3)
fig.text(0.5, 0.928, f"{months[0][1]} {months[0][0]} - {months[-1][1]} {months[-1][0]}   |   {len(all_updates)} updates   |   Generated {today.strftime('%b %d, %Y')}",
         fontsize=13, ha='center', va='top', color='#BDC3C7',
         transform=fig.transFigure, zorder=3, style='italic')

# ========== TOP 5 HIGHLIGHTS ==========
fig.text(0.05, 0.865, 'Top 5 Highlights',
         fontsize=16, fontweight='bold', color='#2C3E50',
         transform=fig.transFigure)

# Pick top 5 most notable updates (GA > Public Preview > Beta > Other, prefer high-impact categories)
def update_priority(u):
    t = u['title'].lower()
    if 'generally available' in t and 'ga' not in t:
        return (0, u['category'])
    if 'public preview' in t:
        return (1, u['category'])
    if 'beta' in t:
        return (2, u['category'])
    return (3, u['category'])

# Get unique high-impact updates across categories
seen_cats = set()
top5 = []
sorted_updates = sorted(all_updates, key=update_priority)
for u in sorted_updates:
    if u['category'] not in seen_cats or len(top5) < 5:
        top5.append(u)
        seen_cats.add(u['category'])
    if len(top5) >= 5:
        break

# If we didn't get 5 unique categories, fill from remaining
if len(top5) < 5:
    for u in sorted_updates:
        if u not in top5:
            top5.append(u)
        if len(top5) >= 5:
            break

highlight_y = 0.83
for i, u in enumerate(top5):
    color = cat_colors.get(u['category'], '#D5D8DC')
    y = highlight_y - (i * 0.028)
    
    # Color dot
    fig.text(0.06, y, '\u25cf', fontsize=10, color=color, va='center',
             transform=fig.transFigure)
    # Title
    title = u['title'][:80] + ('...' if len(u['title']) > 80 else '')
    fig.text(0.085, y, title, fontsize=11, va='center', color='#2C3E50',
             fontweight='bold', transform=fig.transFigure)
    # Date
    fig.text(0.92, y, u['date'][:12], fontsize=9, va='center', ha='right',
             color='#95A5A6', transform=fig.transFigure)
    # Category tag
    fig.text(0.085, y - 0.012, u['category'], fontsize=8, va='center',
             color='#7F8C8D', style='italic', transform=fig.transFigure)

# ========== CATEGORY BAR CHART ==========
fig.text(0.05, 0.66, 'Updates by Category',
         fontsize=16, fontweight='bold', color='#2C3E50',
         transform=fig.transFigure)

ax1 = fig.add_axes([0.08, 0.45, 0.85, 0.18])
cat_counts = Counter(u['category'] for u in all_updates)
cats_sorted = cat_counts.most_common()
cat_names = [c[0] for c in cats_sorted]
cat_vals = [c[1] for c in cats_sorted]
cat_cols = [cat_colors.get(c, '#D5D8DC') for c in cat_names]
bars = ax1.barh(cat_names, cat_vals, color=cat_cols, edgecolor='white', linewidth=1.2, height=0.55)
for bar, val in zip(bars, cat_vals):
    ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=11, fontweight='bold', color='#333')
ax1.set_xlim(0, max(cat_vals) * 1.15)
ax1.invert_yaxis()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.tick_params(axis='y', labelsize=10)
ax1.set_xlabel('Number of Updates', fontsize=10, color='#7F8C8D')
ax1.set_facecolor('#FAFAFA')

# ========== MONTHLY BREAKDOWN ==========
fig.text(0.05, 0.41, 'Monthly Volume',
         fontsize=16, fontweight='bold', color='#2C3E50',
         transform=fig.transFigure)

ax2 = fig.add_axes([0.10, 0.28, 0.35, 0.10])
month_labels_short = [f"{m[1][:3]} {str(m[0])[2:]}" for m in months]
month_counts = [sum(1 for u in all_updates if u['month'] == m[1]) for m in months]
bar_colors_m = ['#FF6B6B', '#4ECDC4', '#45B7D1']
bars2 = ax2.bar(month_labels_short, month_counts, color=bar_colors_m, edgecolor='white', linewidth=1.5, width=0.5)
for bar, count in zip(bars2, month_counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(count), ha='center', fontsize=13, fontweight='bold', color='#333')
ax2.set_ylim(0, max(month_counts) * 1.25)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_facecolor('#FAFAFA')
ax2.tick_params(axis='x', labelsize=9)

# ========== HOW TO USE SECTION ==========
howto_bg = mpatches.FancyBboxPatch((0.015, 0.06), 0.97, 0.16,
    boxstyle="round,pad=0.005", facecolor='#F0F3F4', edgecolor='#D5D8DC', linewidth=1.5,
    transform=fig.transFigure, zorder=2)
fig.patches.append(howto_bg)

fig.text(0.5, 0.205, 'How to Get the Latest Updates',
         fontsize=16, fontweight='bold', ha='center', color='#2C3E50',
         transform=fig.transFigure, zorder=3)

fig.text(0.04, 0.175, '1.  Run This Notebook', fontsize=11, fontweight='bold',
         color='#2C3E50', transform=fig.transFigure, zorder=3)
fig.text(0.04, 0.158, '     Open and run all cells. It dynamically fetches the last 3 months of',
         fontsize=9.5, color='#5D6D7E', transform=fig.transFigure, zorder=3)
fig.text(0.04, 0.143, '     Databricks release notes and renders a fresh one-pager every time.',
         fontsize=9.5, color='#5D6D7E', transform=fig.transFigure, zorder=3)

fig.text(0.04, 0.122, '2.  Weekly AI Task', fontsize=11, fontweight='bold',
         color='#2C3E50', transform=fig.transFigure, zorder=3)
fig.text(0.04, 0.105, '     A scheduled Genie Code task runs every Monday at 9 AM IST, executes',
         fontsize=9.5, color='#5D6D7E', transform=fig.transFigure, zorder=3)
fig.text(0.04, 0.090, '     this notebook, and produces a community-ready summary with the top',
         fontsize=9.5, color='#5D6D7E', transform=fig.transFigure, zorder=3)
fig.text(0.04, 0.075, '     updates, category breakdown, and GA/Beta/deprecation highlights.',
         fontsize=9.5, color='#5D6D7E', transform=fig.transFigure, zorder=3)

fig.text(0.52, 0.175, '3.  Stay Up to Date', fontsize=11, fontweight='bold',
         color='#2C3E50', transform=fig.transFigure, zorder=3)
fig.text(0.52, 0.158, '     This AI-powered workflow ensures you never miss a Databricks update.',
         fontsize=9.5, color='#5D6D7E', transform=fig.transFigure, zorder=3)
fig.text(0.52, 0.143, '     The notebook is idempotent -- no hardcoded data, always fresh.',
         fontsize=9.5, color='#5D6D7E', transform=fig.transFigure, zorder=3)

fig.text(0.52, 0.122, '4.  Share with Your Community', fontsize=11, fontweight='bold',
         color='#2C3E50', transform=fig.transFigure, zorder=3)
fig.text(0.52, 0.105, '     Export the notebook as HTML or screenshot this poster. Share the',
         fontsize=9.5, color='#5D6D7E', transform=fig.transFigure, zorder=3)
fig.text(0.52, 0.090, '     notebook URL so others can run it themselves and get live updates',
         fontsize=9.5, color='#5D6D7E', transform=fig.transFigure, zorder=3)
fig.text(0.52, 0.075, '     anytime they want.',
         fontsize=9.5, color='#5D6D7E', transform=fig.transFigure, zorder=3)

# ========== FOOTER ==========
fig.text(0.5, 0.030, 'Source: docs.databricks.com/aws/en/release-notes/product  |  Powered by Databricks Genie Code',
         fontsize=8, ha='center', color='#BDC3C7', style='italic',
         transform=fig.transFigure)
fig.text(0.5, 0.015, 'Curated by Pratik Bhikadiya  |  This notebook auto-refreshes data on every run',
         fontsize=8, ha='center', color='#BDC3C7',
         transform=fig.transFigure)

plt.savefig('/tmp/databricks_shareable_poster.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
display(fig)
plt.close()